# Bab 14: Support Vector Machine

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab14/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab14")
print("Folder kerja:", os.getcwd())

## Listing 14.1: SVM untuk fungsi AND

`kode/bab14/svm_and.py`

In [ ]:
import numpy as np
from sklearn.svm import SVC

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([-1, -1, -1, 1])
svm = SVC(kernel="linear", C=1e6).fit(X, y)
w, b = svm.coef_[0], svm.intercept_[0]
print(f"w = {w.round(4)}, b = {b:.4f}")
print(f"lebar margin 2/||w|| = {2 / np.linalg.norm(w):.4f}")
print("vektor pendukung:", svm.support_vectors_.tolist())
print("alfa_i * y_i    :", svm.dual_coef_[0].round(4).tolist())
print("y_i (w.x_i + b) :", (y * (X @ w + b)).round(4).tolist())

## Listing 14.2: Pengaruh $C$ pada soft margin

`kode/bab14/soft_margin.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab14')
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from siapkan import muat

df, _, y = muat()
sampel = df.sample(150, random_state=3)
X = StandardScaler().fit_transform(
    sampel[["ketinggian_mdpl", "cacat_primer"]])
ys = np.where(sampel["specialty"] == 1, 1, -1)
for C in [0.01, 1, 100]:
    s = SVC(kernel="linear", C=C).fit(X, ys)
    w, b = s.coef_[0], s.intercept_[0]
    xi = np.maximum(0, 1 - ys * (X @ w + b))
    print(f"C={C:<5g} w=({w[0]:+.2f},{w[1]:+.2f}) "
          f"margin={2 / np.linalg.norm(w):.2f} SV={len(s.support_):3d} "
          f"jumlah xi={xi.sum():5.1f} akurasi={s.score(X, ys):.3f}")
    if C in (0.01, 100):
        nama = "kecil" if C < 1 else "besar"
        with open(f"gambar/bab14/garis_{nama}.dat", "w") as f:
            f.write(f"w1 w2 b\n{w[0]:.4f} {w[1]:.4f} {b:.4f}\n")
        with open(f"gambar/bab14/titik_{nama}.dat", "w") as f:
            f.write("x1 x2 kelas sv\n")
            sv = set(s.support_)
            for i, (a, c) in enumerate(X):
                f.write(f"{a:.3f} {c:.3f} {int(ys[i] > 0)} "
                        f"{int(i in sv)}\n")

## Listing 14.3: Trik kernel dan pengaruh $gamma$

`kode/bab14/kernel.py`

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

x, z = np.array([1.0, 2.0]), np.array([3.0, 0.5])
phi = lambda v: np.array([v[0]**2, v[1]**2, np.sqrt(2) * v[0] * v[1]])
print(f"(x.z)^2          = {(x @ z)**2:.2f}")
print(f"phi(x).phi(z)    = {phi(x) @ phi(z):.2f}")

X, y = make_circles(400, noise=0.08, factor=0.5, random_state=0)
for nama, m in {"linier": SVC(kernel="linear"),
                "polinomial d=2": SVC(kernel="poly", degree=2, coef0=1),
                "RBF": SVC(kernel="rbf", gamma=1.0)}.items():
    s = cross_val_score(m, X, y, cv=5).mean()
    print(f"{nama:15s} akurasi CV {s:.3f}")

for g in [0.1, 1, 10, 100]:
    m = SVC(kernel="rbf", gamma=g, C=1).fit(X, y)
    s = cross_val_score(SVC(kernel="rbf", gamma=g), X, y, cv=5).mean()
    print(f"RBF gamma={g:5g}: akurasi latih {m.score(X, y):.3f}, "
          f"CV {s:.3f}, SV {len(m.support_)}")

## Listing 14.4: Pencarian kisi $C$ dan $gamma$ untuk SVM RBF

`kode/bab14/setel.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab14')
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from siapkan import muat, pra

df, X, y = muat()
pipa = make_pipeline(pra(), SVC(kernel="rbf"))
kisi = {"svc__C": np.logspace(-2, 3, 6),
        "svc__gamma": np.logspace(-4, 1, 6)}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
g = GridSearchCV(pipa, kisi, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X, y)
tab = pd.DataFrame(g.cv_results_["mean_test_score"].reshape(6, 6),
                   index=[f"C={c:g}" for c in kisi["svc__C"]],
                   columns=[f"{v:g}" for v in kisi["svc__gamma"]])
print("AUC CV (baris C, kolom gamma):")
print(tab.round(3).to_string())
b = g.best_params_
print(f"terbaik: C={b['svc__C']:g}, gamma={b['svc__gamma']:g}, "
      f"AUC {g.best_score_:.3f}")
with open("gambar/bab14/kisi.dat", "w") as f:
    for i, c in enumerate(kisi["svc__C"]):
        for j, v in enumerate(kisi["svc__gamma"]):
            f.write(f"{j} {i} {tab.iloc[i, j]:.3f}\n")
        f.write("\n")

## Listing 14.5: SVM multikelas untuk asal biji

`kode/bab14/multikelas.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab14')
import time
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from siapkan import LAB, muat

df, _, _ = muat()
X, y = df[LAB + ["varietas", "proses"]], df["asal"]
pra = ColumnTransformer([("num", StandardScaler(), LAB),
                         ("kat", OneHotEncoder(), ["varietas", "proses"])])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for nama, m in {"satu-lawan-satu": SVC(C=10, gamma=0.05),
                "satu-lawan-sisa": OneVsRestClassifier(
                    SVC(C=10, gamma=0.05))}.items():
    t0 = time.perf_counter()
    yp = cross_val_predict(make_pipeline(pra, m), X, y, cv=cv)
    print(f"{nama}: akurasi {(yp == y).mean():.3f}, "
          f"waktu {time.perf_counter() - t0:.1f} dtk")
kelas = sorted(y.unique())
print("matriks konfusi (satu-lawan-sisa), baris = asal sebenarnya")
print("        " + " ".join(f"{k[:7]:>7s}" for k in kelas))
for k, r in zip(kelas, confusion_matrix(y, yp, labels=kelas)):
    print(f"{k[:7]:7s} " + " ".join(f"{v:7d}" for v in r))

## Listing 14.6: Studi kasus SVM dan keputusan tiga arah

`kode/bab14/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab14')
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from siapkan import muat, pra

df, X, y = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
model = {"logistik": LogisticRegression(C=1, max_iter=2000),
         "SVM linier": SVC(kernel="linear", C=0.1),
         "SVM RBF": SVC(kernel="rbf", C=10, gamma=0.01),
         "random forest": RandomForestClassifier(500, random_state=0)}
print("model           AUC-CV  AUC-uji")
for n, m in model.items():
    p = make_pipeline(pra(), m)
    s = cross_val_score(p, Xl, yl, cv=cv, scoring="roc_auc").mean()
    p.fit(Xl, yl)
    skor = (p.decision_function(Xu) if hasattr(p, "decision_function")
            else p.predict_proba(Xu)[:, 1])
    print(f"{n:15s} {s:7.3f} {roc_auc_score(yu, skor):8.3f}")

svm = make_pipeline(pra(), CalibratedClassifierCV(
    SVC(kernel="linear", C=0.1), cv=5, method="sigmoid"))
svm.fit(Xl, yl)
p = svm.predict_proba(Xu)[:, 1]
yv = yu.to_numpy()
print("\natas  bawah  otomatis  presisi-spec  presisi-kom  cupping")
for atas, bawah in [(0.5, 0.5), (0.8, 0.2), (0.9, 0.1), (0.95, 0.05),
                    (0.95, 0.2)]:
    spec, kom = p >= atas, p <= bawah
    print(f"{atas:4.2f} {bawah:6.2f} {(spec | kom).mean():9.1%} "
          f"{yv[spec].mean():13.3f} {1 - yv[kom].mean():12.3f} "
          f"{1 - (spec | kom).mean():8.1%}")

otomatis = ((p >= 0.95) | (p <= 0.2)).mean()
hemat = otomatis * 500 * 500_000
print(f"\n~500 kiriman/tahun, cupping Rp500 ribu/kiriman; ambang 0,95/0,2"
      f"\nmenghemat ~Rp{hemat / 1e6:.0f} jt per tahun")